# Analysis — both tables, effect sizes, writeup

See `docs/research_proposal.md` §5. Two tables: within-organism (§5.1, filled in separately for Organism A and Organism B) and across-organism (§5.2 — the actual payoff of running two independently-induced organisms rather than one).

Runs entirely locally, like `grade_local.ipynb` -- this notebook only reads two small JSON/JSONL summaries and does plain-Python math/printing, no model or GPU involved. Download `phase4_direction_summary.json` and `phase5_ablation_results.json` from Drive's `artifacts/` into local `artifacts/` (alongside the `labels_*.jsonl` files already there) before running.

In [3]:
import os

os.chdir('/Users/yelyzavetahusieva/Desktop/emergent-misalignment-project')


## Prerequisites checklist

- `artifacts/phase4_direction_summary.json` and `artifacts/phase5_ablation_results.json` downloaded from Drive into local `artifacts/` (produced by `03_extract_directions.ipynb` and `04_cross_ablation.ipynb` in Colab -- those still need GPU, so they stay there).
- `artifacts/labels_{text,mm}_{A,B}.jsonl` already local from `grade_local.ipynb`.


## Load everything from notebooks 03-04

In [4]:
import json
from pathlib import Path

ARTIFACTS = Path('artifacts')
ablation_results = json.loads((ARTIFACTS / 'phase5_ablation_results.json').read_text())
direction_summary = json.loads((ARTIFACTS / 'phase4_direction_summary.json').read_text())
direction_summary, ablation_results


({'within_organism_cosine': {'A': 0.7681363224983215, 'B': 0.7991714477539062},
  'cross_organism_cosine': {'text_A_vs_text_B': 0.8095321655273438,
   'mm_A_vs_mm_B': 0.7542802095413208}},
 {'A': {'cross_mm': [10,
    -7,
    2,
    10,
    20,
    10,
    0,
    20,
    9,
    9,
    16,
    -9,
    -10,
    -20,
    0,
    -7,
    11,
    6,
    20,
    11,
    1,
    5,
    17,
    -7,
    9,
    25,
    5,
    3,
    1,
    5],
   'within_mm': [5,
    0,
    16,
    9,
    21,
    1,
    20,
    31,
    26,
    -5,
    16,
    6,
    -14,
    -7,
    9,
    -16,
    22,
    24,
    6,
    11,
    11,
    -2,
    17,
    -14,
    19,
    0,
    7,
    0,
    -1,
    16],
   'cross_text': [20,
    56,
    -18,
    -4,
    10,
    7,
    23,
    46,
    46,
    -53,
    -10,
    0,
    50,
    0,
    44,
    -30,
    -6,
    10,
    70,
    -38,
    17,
    12,
    37,
    16,
    10,
    52,
    44,
    -16,
    20,
    6],
   'within_text': [7,
    35,
    9,
    -20,
    54,
    60

## §5.1 — Within-organism table (fill in for Organism A, then Organism B)

| | High cosine | Low cosine |
|---|---|---|
| **Strong cross-ablation** | H1: shared direction | Revisit layer/pooling |
| **Weak cross-ablation** | Geometric-only alignment | H0: distinct mechanisms |

"High/low cosine" = `direction_summary['within_organism_cosine'][organism]` vs. the random baseline printed in notebook 03. "Strong/weak cross-ablation" = does that organism's `cross_mm`/`cross_text` mean delta in `ablation_results` approach its `within_mm`/`within_text` mean delta.

In [5]:
for organism in ['A', 'B']:
    cos = direction_summary['within_organism_cosine'][organism]
    deltas = ablation_results[organism]
    mean = lambda xs: sum(xs) / len(xs) if xs else float('nan')
    print(f'--- organism {organism} ---')
    print(f'  cosine(direction_text, direction_mm) = {cos:.4f}')
    print(f'  cross_mm mean delta   = {mean(deltas["cross_mm"]):.1f}   (within_mm baseline: {mean(deltas["within_mm"]):.1f})')
    print(f'  cross_text mean delta = {mean(deltas["cross_text"]):.1f}   (within_text baseline: {mean(deltas["within_text"]):.1f})')
    # fill in: which cell of the table does this organism land in?


--- organism A ---
  cosine(direction_text, direction_mm) = 0.7681
  cross_mm mean delta   = 5.5   (within_mm baseline: 7.8)
  cross_text mean delta = 14.0   (within_text baseline: 14.6)
--- organism B ---
  cosine(direction_text, direction_mm) = 0.7992
  cross_mm mean delta   = -6.9   (within_mm baseline: -6.6)
  cross_text mean delta = -9.4   (within_text baseline: -9.1)


## §5.2 — Across-organism table (the new question)

| Pattern | Interpretation |
|---|---|
| A supports H1 **and** B supports H1 | Strong evidence for a pre-existing, modality-general persona subspace (Nadaf, 2607.21356) — not a training co-occurrence artifact. |
| A supports H1 **but** B supports H0 / no elevated multimodal misalignment | Sharing in A is more likely a training-data-co-occurrence artifact. |
| B shows no elevated multimodal misalignment at all | Purely text-induced misalignment doesn't spontaneously generalize to images — informative prior to any geometry. |
| `direction_text_A` vs `direction_text_B` cosine | Convergence check across independent inductions (same-modality eval) — sanity check that the two organisms are comparable at all. |

In [6]:
cross = direction_summary['cross_organism_cosine']
print(f"cosine(direction_text_A, direction_text_B) = {cross['text_A_vs_text_B']:.4f}")
print(f"cosine(direction_mm_A, direction_mm_B)     = {cross['mm_A_vs_mm_B']:.4f}")
# fill in: which row of the table does this pair of organisms land in?


cosine(direction_text_A, direction_text_B) = 0.8095
cosine(direction_mm_A, direction_mm_B)     = 0.7543


## Bootstrap CIs on judge scores

In [7]:
# bootstrap effect sizes on labels_text_{A,B}.jsonl / labels_mm_{A,B}.jsonl from notebook 02
